# LumenY — OI Directional Model (notebooks_8/03)

**Concept:** Predict whether order imbalance will be significantly higher, flat,
or significantly lower at t+4H and t+12H. Standalone model — no MFE filter.

**Targets** (3-class, Option A):
- `oi_4H`:  0=down  1=flat  2=up  — OI[t+4]  - OI[t] vs ±1 rolling std(24H)
- `oi_12H`: 0=down  1=flat  2=up  — OI[t+12] - OI[t] vs ±1 rolling std(24H)

**Trade logic:** class 2 → long, class 0 → short, class 1 → skip
**Model:** 2 LightGBM multiclass classifiers
**Features:** features_combined
**Train cutoff:** 2024-06-30
**Saved to:** backend/models_9/oi_directional/

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import warnings
import gc
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

FEAT6_DIR  = Path('../backend/data/features_6')
FEAT8_DIR  = Path('../backend/data/features_8')
MODELS_DIR = Path('../backend/models_9/oi_directional')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MAJORS    = ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF', 'USDCAD', 'AUDUSD', 'NZDUSD']
TRAIN_END = '2024-06-30'

OI_STD_WINDOW = 24
TARGET_COLS   = ['oi_4H', 'oi_12H']

print('Ready.')
print(f'OI std window: {OI_STD_WINDOW}H')
print(f'Classes:       0=down  1=flat  2=up')

## 1. Load Data & Build Labels

In [ ]:
dfs_train = []
dfs_test  = []
feature_cols = None

for pair in MAJORS:
    print(f'  {pair}...', flush=True)

    df6 = pd.read_parquet(FEAT6_DIR / f'{pair}_features.parquet')
    df8 = pd.read_parquet(FEAT8_DIR / f'{pair}_geometric.parquet')

    df6.drop(columns=['pair'], errors='ignore', inplace=True)
    df8.drop(columns=['pair'], errors='ignore', inplace=True)

    # align on common index
    idx = df6.index.intersection(df8.index)
    df  = pd.concat([df6.loc[idx], df8.loc[idx]], axis=1)
    del df6, df8; gc.collect()

    # drop duplicate columns if any
    df = df.loc[:, ~df.columns.duplicated()]

    if 'order_imbalance' not in df.columns:
        print(f'    WARNING: order_imbalance not found, skipping')
        del df; gc.collect()
        continue

    if feature_cols is None:
        feature_cols = [c for c in df.columns if c != 'order_imbalance']

    oi         = df['order_imbalance'].copy()
    oi_std     = oi.rolling(OI_STD_WINDOW).std()
    oi_arr     = oi.values
    oi_std_arr = oi_std.values
    n          = len(df)

    label_4H  = np.full(n, np.nan)
    label_12H = np.full(n, np.nan)

    for i in range(n - 12):
        if np.isnan(oi_arr[i]) or np.isnan(oi_std_arr[i]) or oi_std_arr[i] < 1e-10:
            continue
        std = oi_std_arr[i]
        oi0 = oi_arr[i]

        if not np.isnan(oi_arr[i + 4]):
            delta = oi_arr[i + 4] - oi0
            label_4H[i]  = 2 if delta > std else (0 if delta < -std else 1)

        if not np.isnan(oi_arr[i + 12]):
            delta = oi_arr[i + 12] - oi0
            label_12H[i] = 2 if delta > std else (0 if delta < -std else 1)

    df['oi_4H']  = label_4H
    df['oi_12H'] = label_12H

    num_cols = [c for c in df.columns
                if c not in TARGET_COLS and df[c].dtype != object]
    df[num_cols] = df[num_cols].ffill().fillna(0).astype(np.float32)

    dfs_train.append(df[df.index <= TRAIN_END])
    dfs_test.append(df[df.index >  TRAIN_END])
    del df, label_4H, label_12H; gc.collect()

df_train = pd.concat(dfs_train).sort_index(); del dfs_train; gc.collect()
df_test  = pd.concat(dfs_test).sort_index();  del dfs_test;  gc.collect()

feature_cols = [c for c in df_train.columns
                if c not in TARGET_COLS and df_train[c].dtype != object]

print(f'Features:   {len(feature_cols)}')
print(f'Train rows: {len(df_train):,}')
print(f'Test rows:  {len(df_test):,}')
print(f'RAM train:  {df_train.memory_usage(deep=False).sum()/1024**2:.0f} MB')
print(f'\nLabel distribution (train):')
for col in TARGET_COLS:
    valid  = df_train[col].notna()
    counts = df_train.loc[valid, col].value_counts().sort_index()
    total  = valid.sum()
    print(f'  {col}: down={counts.get(0,0)/total:.1%}  flat={counts.get(1,0)/total:.1%}  up={counts.get(2,0)/total:.1%}  (n={total:,})')

## 2. Prepare Arrays

In [ ]:
valid_train = df_train['oi_4H'].notna() & df_train['oi_12H'].notna()
X_train   = df_train.loc[valid_train, feature_cols].values.astype(np.float32)
y4_train  = df_train.loc[valid_train, 'oi_4H'].astype(np.int8).values
y12_train = df_train.loc[valid_train, 'oi_12H'].astype(np.int8).values
del df_train; gc.collect()

valid_test = df_test['oi_4H'].notna() & df_test['oi_12H'].notna()
X_test    = df_test.loc[valid_test, feature_cols].values.astype(np.float32)
y4_test   = df_test.loc[valid_test, 'oi_4H'].astype(np.int8).values
y12_test  = df_test.loc[valid_test, 'oi_12H'].astype(np.int8).values
del df_test; gc.collect()

print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')
print(f'RAM X_train: {X_train.nbytes/1024**2:.0f} MB')


## 3. Walk-Forward CV

In [ ]:
def walk_forward_splits(n, n_splits=5, test_ratio=0.1):
    splits = []
    test_size = int(n * test_ratio)
    for i in range(n_splits):
        test_start = int(n * 0.5) + i * (int(n * 0.5) // n_splits)
        test_end   = test_start + test_size
        if test_end > n: break
        splits.append((list(range(0, test_start)), list(range(test_start, test_end))))
    return splits

splits = walk_forward_splits(len(X_train))
print(f'Splits: {len(splits)}')
for i, (tr, te) in enumerate(splits):
    print(f'  Fold {i+1}: train {len(tr):,} | val {len(te):,}')


## 4. Train

In [ ]:
def get_lgbm_params():
    return {
        'objective':         'multiclass',
        'num_class':         3,
        'metric':            'multi_logloss',
        'boosting_type':     'gbdt',
        'n_estimators':      3000,
        'learning_rate':     0.02,
        'num_leaves':        64,
        'max_depth':         6,
        'min_child_samples': 50,
        'feature_fraction':  0.7,
        'bagging_fraction':  0.8,
        'bagging_freq':      5,
        'reg_alpha':         0.1,
        'reg_lambda':        0.1,
        'random_state':      42,
        'n_jobs':            -1,
        'verbose':           -1,
        'device':            'gpu',
    }

cv_results = {}
best_iters = {}

for name, y_tr, y_te in [('oi_4H', y4_train, y4_test), ('oi_12H', y12_train, y12_test)]:
    print(f'\nCV — {name}')
    fold_aucs = []; fold_iters = []

    for fold, (tr_idx, te_idx) in enumerate(splits):
        X_tr, y_trr = X_train[tr_idx], y_tr[tr_idx]
        X_te, y_tee = X_train[te_idx], y_tr[te_idx]
        if len(np.unique(y_tee)) < 2:
            print(f'  Fold {fold+1}: skipped'); continue

        model = lgb.LGBMClassifier(**get_lgbm_params())
        model.fit(X_tr, y_trr, eval_set=[(X_te, y_tee)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])

        probs = model.predict_proba(X_te)
        y_bin = label_binarize(y_tee, classes=[0,1,2])
        auc   = np.mean([roc_auc_score(y_bin[:,c], probs[:,c])
                         for c in range(3) if y_bin[:,c].sum() > 0])
        fold_aucs.append(auc)
        fold_iters.append(model.best_iteration_)
        print(f'  Fold {fold+1}: AUC={auc:.4f}  iters={model.best_iteration_}')
        del model, X_tr, y_trr, X_te, y_tee; gc.collect()

    cv_results[name] = float(np.mean(fold_aucs)) if fold_aucs else 0.0
    best_iters[name] = int(np.mean(fold_iters))  if fold_iters else 500
    print(f'  Mean AUC: {cv_results[name]:.4f}  avg iters: {best_iters[name]}')

## 5. Train Final Models & Save

In [ ]:
models = {}
for name, y_arr in [('oi_4H', y4_train), ('oi_12H', y12_train)]:
    iters = best_iters.get(name, 500)
    print(f'  {name}: {iters} iters...', flush=True)
    params = {**get_lgbm_params(), 'n_estimators': iters}
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_arr)
    models[name] = model
    del model; gc.collect()

joblib.dump({
    'models':        models,
    'feature_cols':  feature_cols,
    'target_cols':   TARGET_COLS,
    'train_end':     TRAIN_END,
    'oi_std_window': OI_STD_WINDOW,
    'cv_auc':        cv_results,
    'best_iters':    best_iters,
    'classes':       {0: 'down', 1: 'flat', 2: 'up'},
}, MODELS_DIR / 'model_oi_directional.joblib')

size_mb = (MODELS_DIR / 'model_oi_directional.joblib').stat().st_size / 1024**2
print(f'Saved ({size_mb:.1f} MB)')
del X_train, y4_train, y12_train; gc.collect()

## 6. Test Evaluation

In [ ]:
print('Test set evaluation:')
print(f'  {"Target":<10} {"AUC":>8} {"CV AUC":>8}  Class distribution')
print('  ' + '-'*55)

for name, y_arr in [('oi_4H', y4_test), ('oi_12H', y12_test)]:
    probs = models[name].predict_proba(X_test)
    y_bin = label_binarize(y_arr, classes=[0,1,2])
    auc   = np.mean([roc_auc_score(y_bin[:,c], probs[:,c])
                     for c in range(3) if y_bin[:,c].sum() > 0])
    preds = np.argmax(probs, axis=1)
    dist  = ' | '.join([f'{c}:{(preds==c).mean():.1%}' for c in range(3)])
    print(f'  {name:<10} {auc:>8.4f} {cv_results.get(name,0):>8.4f}  pred:[{dist}]')

print(f'\nCalibration — oi_4H:')
probs_4H = models['oi_4H'].predict_proba(X_test)
print(f'  {"Class":<8} {"Conf>":<6} {"N":>6} {"Match%":>8} {"Base%":>8}')
print('  ' + '-'*40)
for cls, cls_name in [(0,'down'),(2,'up')]:
    base = (y4_test == cls).mean()
    for thresh in [0.35, 0.40, 0.45, 0.50]:
        mask = probs_4H[:, cls] > thresh
        if mask.sum() < 20: continue
        acc = (y4_test[mask] == cls).mean()
        print(f'  {cls_name:<8} {thresh:<6.2f} {mask.sum():>6,} {acc:>8.1%} {base:>8.1%}')
